# Fine-tune the small model on financial-report questions, then test it

This notebook trains Gemma 3 4B on about 1,000 solved questions from the public FinQA training split (banks, insurers and payment companies), then tests it on the 297 test questions it has never seen.

**Before you start**

1. In the menu choose **Runtime > Change runtime type**, pick **T4 GPU**, and save.
2. Choose **Runtime > Run all**.
3. Keep this tab open until a file named `fin_finetune_results.zip` downloads. That file is the result.

Nothing here needs an account, a key or a payment.

## 1. Install and get the project

In [ ]:
%%capture
!pip install -q unsloth
!rm -rf /content/slm-opti && git clone -q --depth 1 -b finance-qa https://github.com/eshwarpotturi/slm-opti /content/slm-opti

## 2. Load the model

In [ ]:
import os, json, time, glob, zipfile
os.chdir("/content/slm-opti")
import sys
sys.path.insert(0, "/content/slm-opti/fin")
import torch
assert torch.cuda.is_available(), "No GPU. Choose Runtime > Change runtime type > T4 GPU, then Runtime > Run all."

from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(
    model_name="unsloth/gemma-3-4b-it",   # the same Gemma 3 4B weights, hosted without a sign-in
    max_seq_length=3072,
    load_in_4bit=True,
    full_finetuning=False,
)
print("Model loaded.")

In [ ]:
import stack, run

def _msgs(messages):
    return [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in messages]

@torch.no_grad()
def hf_chat(messages, temperature=0, max_tokens=400):
    inputs = tokenizer.apply_chat_template(_msgs(messages), add_generation_prompt=True, tokenize=True,
                                           return_dict=True, return_tensors="pt").to("cuda")
    kw = dict(max_new_tokens=max_tokens, do_sample=temperature > 0)
    if temperature > 0:
        kw["temperature"] = temperature
    out = model.generate(**inputs, **kw)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

stack.HF_CHAT = hf_chat
print("Check, 2 + 2 =", hf_chat([{"role": "user", "content": "What is 2 + 2? Reply with the number only."}], max_tokens=5))

## 3. Train

Each example is a report extract and a question; the answer to learn is the formula. One pass over about 1,000 examples.

In [ ]:
from datasets import Dataset

N_TRAIN = 1200
pool = [t for t in json.load(open("fin/data/train.json"))
        if (isinstance(t["answer"], str) or (t["formula"] and run._safe(t))) and len(run.render(t).split()) <= 900]
train = pool[:N_TRAIN]
rows = []
for t in train:
    target = t["answer"] if isinstance(t["answer"], str) else t["formula"]
    msgs = [{"role": "user", "content": [{"type": "text", "text": run.TUNED + "\n\n" + run.ask(t)}]},
            {"role": "assistant", "content": [{"type": "text", "text": target}]}]
    rows.append({"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False).removeprefix("<bos>")})
ds = Dataset.from_list(rows)
lengths = [len(tokenizer(r["text"])["input_ids"]) for r in rows[:100]]
print(len(rows), "training examples; of the first 100 the longest is", max(lengths), "tokens")
print(rows[0]["text"][-300:])

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407,
)

from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=1, gradient_accumulation_steps=8,
        warmup_steps=5, num_train_epochs=1, learning_rate=2e-4,
        logging_steps=5, optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="linear",
        seed=3407, report_to="none", output_dir="outputs",
    ),
)
# learn only from the answer, not from the long prompt
trainer = train_on_responses_only(trainer, instruction_part="<start_of_turn>user\n", response_part="<start_of_turn>model\n")
t0 = time.time()
os.makedirs('fin/results', exist_ok=True)
stats = trainer.train()
json.dump({"minutes": round((time.time() - t0) / 60, 1), "examples": len(rows), "log": trainer.state.log_history},
          open("fin/results/finetune_training_log.json", "w"))
print("Training took %.0f minutes." % ((time.time() - t0) / 60))

## 4. Save the tuned model first

A small download, so the training is not lost if the tab closes during the test.

In [ ]:
model.save_pretrained("/content/fin_adapter")
import shutil
shutil.make_archive("/content/fin_adapter", "zip", "/content/fin_adapter")
from google.colab import files
files.download("/content/fin_adapter.zip")

## 5. The tuned model on the unseen test

In [ ]:
model.eval()
print("Check, 2 + 2 =", hf_chat([{"role": "user", "content": "What is 2 + 2? Reply with the number only."}], max_tokens=5))
run.main(["hf:gemma-3-4b-tuned", "--layers", "0", "--tag", "t", "--tuned", "--workers", "1", "--budget", "100000"])
run.main(["hf:gemma-3-4b-tuned", "--layers", "2", "--tag", "t", "--tuned", "--workers", "1", "--budget", "100000"])

## 6. Results

In [ ]:
with zipfile.ZipFile("/content/fin_finetune_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob("fin/results/gemma-3-4b-tuned*.jsonl") + ["fin/results/finetune_training_log.json"]:
        if os.path.exists(f):
            z.write(f)
files.download("/content/fin_finetune_results.zip")